# MorphoMNIST: VAE and Geodesic Symmetry Experiment

This notebook trains and evaluates two models on the **MorphoMNIST** dataset:

| Model | Description |
|-------|-------------|
| **Vanilla VAE** | Standard convolutional VAE trained with ELBO |
| **GSVAE** | VAE with geodesic-symmetry regularisation that encourages a symmetric latent space around a learned anchor point |

The notebook proceeds through the following steps:
1. Data loading (r2r split)
2. VAE training / checkpoint loading
3. Latent-space Riemannian metric computation via Jacobian
4. Geodesic model training (vanilla VAE)
5. GSVAE training
6. Side-by-side qualitative comparison
7. Quantitative symmetry analysis with morphological measurements

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from functorch import vmap, jacfwd

from morphomnist import io
from vae import VAE
from geodesic import GeodesicMaker, geodesic
from dataset import MorphoMNISTDataset, r2r

## 1. Reproducibility

Fix all random seeds (Python, NumPy, PyTorch, CUDA) to ensure reproducibility across runs.

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"
random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
np.random.seed(42)

## 2. Data Loading

Load the full MorphoMNIST dataset (train + test), concatenate it, then apply the **r2r** (random-to-random) split so that training and test sets are disjoint subsets of the full dataset.

In [ ]:
train_set = MorphoMNISTDataset(train=True)
test_set = MorphoMNISTDataset(train=False)
concat_set = MorphoMNISTDataset(train_set=train_set, test_set=test_set)
train_set, test_set = r2r(concat_set)
train_loader = DataLoader(dataset = train_set, batch_size=100, shuffle=True)
test_loader = DataLoader(dataset = test_set, batch_size=100, shuffle=False)

## 3. Model Initialization

Instantiate the **VAE** (encoder + decoder) and the **GeodesicMaker** (the network that predicts per-sample geodesic curves) and move them to the GPU.

In [ ]:
vae_model = VAE()
vae_model = vae_model.cuda()
geodesic_maker = GeodesicMaker().cuda()

## 4. VAE Training

Define the **ELBO loss** (BCE reconstruction + KL divergence), training and evaluation loops, then run 50 epochs (or skip if a saved checkpoint exists).  
The geodesic-regularisation branch inside `train()` is currently commented out — it is used only when training the geodesic model jointly with the VAE.

In [ ]:
if os.path.exists('vae_model.pth'):
    vae_model.load_state_dict(torch.load('vae_model.pth'))
    vae_model.eval()
    print("Pretrained model loaded successfully.")
    loaded = True
else:
    print("Pretrained model not found.")
    loaded = False


In [ ]:
optimizer = optim.Adam(vae_model.parameters(), lr=1e-3)
geo_optim = optim.Adam(geodesic_maker.parameters(), lr=1e-3)
def loss_function(output, x, mu, logvar):
    bce = nn.functional.binary_cross_entropy(output, x, reduction="sum")
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + kld

def geodesic_loss_function(z):
    batch_size = z.size(0)
    w1, w2, b1, b2 = geodesic_maker(z)
    
    zero = torch.zeros((z.size(0), 1, 1)).cuda()
    one = torch.ones((z.size(0), 1, 1)).cuda()
    
    energy = 0
    
    
    gamma_zero, mu_zero, logvar_zero = geodesic(zero, w1, w2, b1, b2, vae_model.encoder)
    gamma_one, mu_one, logvar_one = geodesic(one, w1, w2, b1, b2, vae_model.encoder)

    gamma_old = gamma_zero
    mu_old = mu_zero
    logvar_old = logvar_zero

    for i in range(1, 11):
        p = one * (i / 10)
        gamma_p, mu_p, logvar_p = geodesic(p, w1, w2, b1, b2, vae_model.encoder)
        energy += torch.pow(mu_old - mu_p, 2) + (logvar_old.exp() + logvar_p.exp())
        gamma_old, mu_old, logvar_old = gamma_p, mu_p, logvar_p
        
    geodesic_loss = (
        nn.functional.l1_loss(gamma_zero, geodesic_maker.anchor.unsqueeze(0).repeat(batch_size, 1), reduction="sum")
        + nn.functional.l1_loss(gamma_one, z, reduction="sum")
        + energy.sum() / 2
    )
        
    return geodesic_loss / batch_size

def train(epoch):
    vae_model.train()
    train_loss = 0
    train_geodesic_loss = 0
    for batch_idx, (data, _, _) in enumerate(train_loader):
        data = data.cuda()
        optimizer.zero_grad()
        
        output, z, mu, logvar = vae_model(data)
        loss = loss_function(output, data, mu, logvar)
        
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        
        """geodesic_loss = geodesic_loss_function(z.detach().clone())
        
        geodesic_loss.backward()
        train_geodesic_loss += geodesic_loss.item()
        geo_optim.step()"""
        
        
    train_loss /= len(train_loader) * 100
    print("Train epoch: {},\tLoss: {:.6f}".format(epoch, train_loss))

def test():
    vae_model.eval()
    test_loss = 0
    with torch.no_grad():
        for batch_idx, (data, _, _) in enumerate(test_loader):
            data = data.cuda()
            output, z, mu, logvar = vae_model(data)
            test_loss += loss_function(output, data, mu, logvar).item()
            
        test_loss /= len(test_loader) * 100
    print("test loss", test_loss)

In [ ]:
for epoch in range(50):
    if loaded:
        print("skip training")
        break
    train(epoch)
    test()


In [ ]:
if loaded:
    pass
else:
    torch.save(vae_model.state_dict(), 'vae_model.pth')


## 5. Latent-Space Riemannian Metric

Compute the **Riemannian metric tensor** of the decoder manifold at each point of a 200×200 grid in latent space.

$$G(z) = J(z)^\top J(z), \quad \text{where } J(z) = \frac{\partial \text{decoder}(z)}{\partial z}$$

The square-root of the determinant of $G$ (the *volume indicator*) is plotted as a heatmap, showing how the decoder stretches the latent space — peaks correspond to high-curvature regions.

`jacfwd` + `vmap` (from `functorch`) are used for efficient batched Jacobian computation.

In [ ]:
latent_x = np.linspace(-5, 5, 200)
latent_y = np.linspace(-5, 5, 200)
xv, yv = np.meshgrid(latent_x, latent_y)
xtensor = torch.tensor(xv.reshape(-1))
ytensor = torch.tensor(yv.reshape(-1))

In [ ]:
ztensor = torch.stack((xtensor, ytensor), dim=1)
ztensor = torch.tensor(ztensor, dtype=torch.float32)
ztensor.shape

In [ ]:
vae_model = vae_model.cpu()
jacobian = vmap(jacfwd(vae_model.decoder))(ztensor)
vae_model = vae_model.cuda()

In [ ]:
jacobian.shape

In [ ]:
jacobian = jacobian.view(40000, 28*28, 2)

In [ ]:
metric = torch.bmm(jacobian.permute(0, 2, 1), jacobian)

In [ ]:
indi = torch.sqrt(torch.linalg.det(metric).clamp_(min=1e-5)).detach().cpu().numpy()

In [ ]:
indi.shape

In [ ]:
indi = indi.reshape(200, 200)

In [ ]:
h = plt.contourf(latent_x, latent_y, np.log(indi), extend="both")
anchor = geodesic_maker.anchor.detach().clone().cpu().numpy()
plt.scatter(anchor[0], anchor[1],c="r")
plt.colorbar()
plt.show()

## 6. Collecting Latent Codes

Encode all training and test images into the 2D latent space.  
Randomly subsample 18 000 training and 2 000 test latent codes for visualisation (to avoid overloaded scatter plots).

In [ ]:
anchor

In [ ]:
z_list = list()
vae_model.eval()
with torch.no_grad():
    for batch_idx, (data, _, _) in enumerate(test_loader):
        data = data.cuda()
        output, z, mu, logvar = vae_model(data)
        z_list.append(z.detach().cpu().numpy())

In [ ]:
train_z_list = list()
vae_model.eval()
with torch.no_grad():
    for batch_idx, (data, _, _) in enumerate(train_loader):
        data = data.cuda()
        output, z, mu, logvar = vae_model(data)
        train_z_list.append(z.detach().cpu().numpy())
z_train = np.concatenate(train_z_list)

In [ ]:
zs = np.concatenate(z_list, axis=0)
zs.shape

In [ ]:
random_z_test = np.random.permutation(zs)[:2000]
random_z_train = np.random.permutation(z_train)[:18000]

In [ ]:
plt.figure(figsize=(10,6))
plt.contourf(latent_x, latent_y, np.log(indi), extend="both")
anchor = geodesic_maker.anchor.detach().clone().cpu().numpy()
plt.scatter(anchor[0], anchor[1],c="r")
#plt.scatter(zs[:,0], zs[:,1], c='g', alpha=0.5, s=0.5)
#plt.scatter()
plt.scatter(random_z_test[:,0], random_z_test[:,1], c='b', alpha=0.5, s=0.5)
plt.scatter(random_z_train[:,0], random_z_train[:,1], c='g', alpha=0.5, s=0.5)
plt.colorbar()
plt.show()

## 7. Geodesic Model — Training (Vanilla VAE)

The `GeodesicMaker` network predicts a **Bézier-like curve** through the latent space connecting a query point to the learned anchor.  
The loss minimises:
- Endpoint constraints (curve starts at anchor, ends at query point)
- Integrated kinetic energy along the curve under the pull-back Riemannian metric

Load checkpoint if available; otherwise train for 100 iterations.

In [ ]:
geo_loaded = False

if os.path.exists('geodesic.pth'):
    geodesic_maker.load_state_dict(torch.load('geodesic.pth'))
    geodesic_maker.eval()
    print("Pretrained geodesic model loaded successfully.")
    geo_loaded = True
else:
    print("Pretrained geodesic model not found.")
    geo_loaded = False


In [ ]:
def geodesic_training():
    vae_model.eval()
    train_geodesic_loss = 0
    for batch_idx, (data, _, _) in enumerate(train_loader):
        data = data.cuda()
        geo_optim.zero_grad()
        
        output, z, mu, logvar = vae_model(data)
        
        geodesic_loss = geodesic_loss_function(z.detach().clone())
        
        geodesic_loss.backward()
        train_geodesic_loss += geodesic_loss.item()
        geo_optim.step()
        
        
    print("Geodesic loss {:.6f}".format(train_geodesic_loss))

In [ ]:
w1, w2, b1, b2 = geodesic_maker(torch.randn(100,2).cuda())

zero = torch.zeros((100, 1, 1)).cuda()
one = torch.ones((100, 1, 1)).cuda()

energy = 0

gamma_zero, mu_zero, logvar_zero = geodesic(zero, w1, w2, b1, b2, vae_model.encoder)
gamma_one, mu_one, logvar_one = geodesic(one, w1, w2, b1, b2, vae_model.encoder)

In [ ]:
for i in range(100):
    if geo_loaded:
        break
    geodesic_training()
    
if geo_loaded:
    pass
else:
    torch.save(geodesic_maker.state_dict(), 'geodesic.pth')

## 8. Geodesic Curve Visualisation (Vanilla VAE)

Sample a reference point from the training set, encode it, then trace the geodesic path from the anchor back to that point by stepping the curve parameter from 1.0 → 0.0.  
Plot the resulting curve (magenta) over the latent-space metric heatmap.

In [ ]:
x0 = train_set[80][0].cuda()
_, z0, _, _ = vae_model(x0.unsqueeze(0))

In [ ]:
geo_list = list()
one = torch.ones((1)).cuda()

w1, w2, b1, b2 = geodesic_maker(z0)

for i in range(100):
    p = one * (50 - i) / 50
    z, _, _ = geodesic(p, w1, w2, b1, b2, vae_model.encoder)
    geo_list.append(z)

In [ ]:
geo_tensor = torch.stack(geo_list, axis=0)
geo_np = geo_tensor.detach().cpu().numpy()
geo_np.shape

In [ ]:
geo_np[1]

In [ ]:
np.log(indi).max()

In [ ]:
plt.figure(figsize=(15,13))
plt.contour(latent_x, latent_y, np.log(indi), levels=[0, 3], colors='k')
#plt.contourf(latent_x, latent_y, np.log(indi), levels=5, extend="both")
cont = plt.contourf(latent_x, latent_y, np.log(indi), extend="both", cmap="gist_gray")
anchor = geodesic_maker.anchor.detach().clone().cpu().numpy()
plt.scatter(anchor[0], anchor[1],c="r")
#plt.scatter(zs[:,0], zs[:,1], c='b', alpha=0.5, s=0.7)
plt.scatter(random_z_train[:,0], random_z_train[:,1], c='g', alpha=0.5, s=0.5)
plt.scatter(random_z_test[:,0], random_z_test[:,1], c='b', alpha=0.5, s=0.5)
plt.plot(geo_np[:, 0], geo_np[:,1], 'm')
plt.plot([geo_np[1,0], 2*anchor[0]-geo_np[1,0]], [geo_np[1,1],2*anchor[1]-geo_np[1,1]], 'k-.')
plt.colorbar(cont)
plt.show()

In [ ]:
plt.figure(figsize=(15,10))
plt.contour(latent_x, latent_y, np.log(indi), levels=[0, 4], colors='k')
#plt.contourf(latent_x, latent_y, np.log(indi), levels=5, extend="both")
cont = plt.contourf(latent_x, latent_y, np.log(indi), extend="both", cmap='gist_gray')
anchor = geodesic_maker.anchor.detach().clone().cpu().numpy()
plt.scatter(anchor[0], anchor[1],c="r")
#plt.scatter(zs[:,0], zs[:,1], c='b', alpha=0.5, s=0.7)
plt.scatter(random_z_train[:, 0], random_z_train[:, 1], c='g', alpha=0.5, s=0.5)
plt.scatter(random_z_test[:, 0], random_z_test[:, 1], c='b', alpha=0.5, s=0.5)
plt.plot(geo_np[:, 0], geo_np[:,1], 'm')
plt.plot([geo_np[1,0], 2*anchor[0]-geo_np[1,0]], [geo_np[1,1],2*anchor[1]-geo_np[1,1]], 'k-.')
plt.colorbar(cont)
plt.axis([-2.5, 2.5, -2.5, 2.5])
plt.show()

## 9. GSVAE — Geodesic-Symmetry VAE

Import, instantiate, and train the **GSVAE** model.  
`GSVAE` extends the standard VAE with a geodesic-symmetry regularisation term that encourages the latent space to be symmetric about a shared anchor point:

$$\mathcal{L}_{\text{GSVAE}} = \mathcal{L}_{\text{ELBO}} + \lambda_{\text{GS}} \cdot \mathcal{L}_{\text{GS}}$$

where $\mathcal{L}_{\text{GS}}$ penalises asymmetric geodesic paths.

In [ ]:
from gsvae import GSVAE

In [ ]:
gsvae = GSVAE()
gsoptim = optim.Adam(gsvae.parameters(), lr=1e-3)

In [ ]:
gs_loaded = False

if os.path.exists('gsvae.pth'):
    gsvae.load_state_dict(torch.load('gsvae.pth'))
    gsvae.eval()
    print("Pretrained gsvae model loaded successfully.")
    gs_loaded = True
else:
    print("Pretrained gsvae model not found.")
    gs_loaded = False


In [ ]:
def gstrain(epoch):
    gsvae.train()
    train_loss = 0
    for batch_idx, (data, _, _) in enumerate(train_loader):
        data = data.cuda()
        gsoptim.zero_grad()
        
        (rec, kld, gs), output, z, mu, logvar = gsvae(data)
        loss = (rec + kld + 5 * gs)
        loss.backward()
        train_loss += loss.item()
        
        gsoptim.step()
        
    train_loss /= len(train_loader) * 100
    print("Train epoch: {},\tLoss: {:.6f}".format(epoch, train_loss))
    #print("Geodesic loss {:.6f}".format(train_geodesic_loss))

def gstest():
    gsvae.eval()
    test_loss = 0
    rec_loss = 0
    kld_loss = 0
    gs_loss = 0
    with torch.no_grad():
        for batch_idx, (data, _, _) in enumerate(test_loader):
            data = data.cuda()
            (rec, kld, gs), output, z, mu, logvar = gsvae(data)
            test_loss += (rec + kld + 5 * gs).item()
            rec_loss += rec.item()
            kld_loss += kld.item()
            gs_loss += gs.item()
            
            
    test_loss /= len(test_loader) * 100
    rec_loss /= len(test_loader) * 100
    kld_loss /= len(test_loader) * 100
    gs_loss /= len(test_loader) * 100
    
    print("test loss", test_loss)
    print("rec loss", rec_loss)
    print("kld loss", kld_loss)
    print("gs loss", gs_loss )

In [ ]:
gsvae = gsvae.cuda()

In [ ]:
for i in range(50):
    if gs_loaded:
        print("skip training")
        break
    gstrain(i)
    gstest()
    
if gs_loaded:
    pass
else:
    torch.save(gsvae.state_dict(), 'gsvae.pth')

## 10. GSVAE — Latent Space Metric & Geodesic Curves

Compute the Riemannian metric of the **GSVAE decoder** over the same 200×200 grid.  
Then trace geodesic curves from two different points (train index 80 and test index 10) to the anchor, using the GSVAE's built-in `gamma` module.

In [ ]:
gsvae = gsvae.cpu()
gsjacobian = vmap(jacfwd(gsvae.vae.decoder))(ztensor)
gsjacobian = gsjacobian.view(40000, 28*28, 2)
gsmetric = torch.bmm(gsjacobian.permute(0, 2, 1), gsjacobian)
gsindi = torch.sqrt(torch.linalg.det(gsmetric).clamp_(min=1e-5)).detach().cpu().numpy()
gsindi = gsindi.reshape(200, 200)

In [ ]:
gsvae = gsvae.cuda()
geo_list0 = list()
x0 = train_set[80][0].cuda()
_, _, z0, _, _ = gsvae(x0)
one = torch.ones((1)).cuda()

w1, w2, b1, b2 = gsvae.gamma(z0)

for i in range(100):
    p = one * (50 - i) / 50
    z, _, _ = geodesic(p, w1, w2, b1, b2, gsvae.vae.encoder)
    geo_list0.append(z)

In [ ]:
geo_list1 = list()
x1 = test_set[10][0].cuda()
_, _, z1, _, _ = gsvae(x1)
one = torch.ones((1)).cuda()

w1, w2, b1, b2 = gsvae.gamma(z1)

for i in range(100):
    p = one * (50 - i) / 50
    z, _, _ = geodesic(p, w1, w2, b1, b2, gsvae.vae.encoder)
    geo_list1.append(z)

In [ ]:
geo_tensor0 = torch.stack(geo_list0, axis=0)
geo_np0 = geo_tensor0.detach().cpu().numpy()
geo_np0.shape

In [ ]:
geo_tensor1 = torch.stack(geo_list1, axis=0)
geo_np1 = geo_tensor1.detach().cpu().numpy()
geo_np1.shape

In [ ]:
z_list = list()
gsvae.eval()
with torch.no_grad():
    for batch_idx, (data, _, _) in enumerate(test_loader):
        data = data.cuda()
        loss, output, z, mu, logvar = gsvae(data)
        z_list.append(z.detach().cpu().numpy())
gszs = np.concatenate(z_list, axis=0)

train_z_list = list()
gsvae.eval()
with torch.no_grad():
    for batch_idx, (data, _, _) in enumerate(train_loader):
        data = data.cuda()
        loss, output, z, mu, logvar = gsvae(data)
        train_z_list.append(z.detach().cpu().numpy())
gsz_train = np.concatenate(train_z_list)

In [ ]:
random_gsz_test = np.random.permutation(gszs)[:2000]
random_gsz_train = np.random.permutation(gsz_train)[:18000]

In [ ]:
plt.figure(figsize=(15,13))
plt.contour(latent_x, latent_y, np.log(gsindi), levels=[0, 4], colors='k')
#plt.contourf(latent_x, latent_y, np.log(indi), levels=3, extend="both")
cont = plt.contourf(latent_x, latent_y, np.log(gsindi), extend="both", cmap="gist_gray")
#plt.contour(latent_x, latent_y, np.log(indi), extend="both", colors='k')
anchor = gsvae.gamma.anchor.detach().clone().cpu().numpy()
plt.scatter(anchor[0], anchor[1],c="r")
#plt.scatter(gszs[:,0], gszs[:,1], c='b', alpha=0.5, s=0.7)
plt.scatter(random_gsz_train[:,0], random_gsz_train[:,1], c='g', alpha=0.5, s=0.5)
plt.scatter(random_gsz_test[:,0], random_gsz_test[:,1], c='b', alpha=0.5, s=0.5)
plt.plot(geo_np0[:, 0], geo_np0[:,1], 'm-')
#plt.plot(geo_np)
plt.plot([geo_np0[0,0], 2*anchor[0]-geo_np0[0,0]], [geo_np0[0,1], 2*anchor[1]-geo_np0[0,1]], 'k-.')
#plt.plot(geo_np1[:, 0], geo_np1[:,1], 'y')
#plt.plot([geo_np1[0,0], 2*anchor[0]-geo_np1[0,0]], [geo_np1[0,1], 2*anchor[1]-geo_np1[0,1]], 'c')
plt.colorbar(cont)
plt.show()

In [ ]:
plt.figure(figsize=(15,10))
plt.contour(latent_x, latent_y, np.log(gsindi), levels=[0, 4], colors='k')
#plt.contourf(latent_x, latent_y, np.log(indi), levels=3, extend="both")
cont = plt.contourf(latent_x, latent_y, np.log(gsindi), extend="both", cmap="gist_gray")
#plt.contour(latent_x, latent_y, np.log(indi), extend="both", colors='k')
anchor = gsvae.gamma.anchor.detach().clone().cpu().numpy()
plt.scatter(anchor[0], anchor[1],c="r")
plt.scatter(random_gsz_train[:,0], random_gsz_train[:,1], c='g', alpha=0.5, s=0.5)
plt.scatter(random_gsz_test[:,0], random_gsz_test[:,1], c='b', alpha=0.5, s=0.5)
plt.plot(geo_np0[:, 0], geo_np0[:,1], 'm-')
#plt.plot(geo_np)
plt.plot([geo_np0[0,0], 2*anchor[0]-geo_np0[0,0]], [geo_np0[0,1], 2*anchor[1]-geo_np0[0,1]], 'k-.')
plt.plot(geo_np1[:, 0], geo_np1[:,1], 'y')
plt.plot([geo_np1[0,0], 2*anchor[0]-geo_np1[0,0]], [geo_np1[0,1], 2*anchor[1]-geo_np1[0,1]], 'c')
plt.colorbar(cont)
plt.axis([-2.5, 2.5, -2.5, 2.5])
plt.show()

## 11. Side-by-Side Comparison

Overlay a single geodesic path from each model on their respective metric heatmaps for direct visual comparison.  
Vanilla VAE geodesics may cross low-density regions, while GSVAE geodesics are expected to stay in high-density (symmetric) regions.

In [ ]:
vanilla_geodesic_list = list()
vae_model.eval()
z0 = torch.tensor(random_z_test[50]).unsqueeze(0)
w1, w2, b1, b2 = geodesic_maker(torch.tensor(z0).cuda())
one = torch.ones((1)).cuda()

for i in range(100):
    p = one * i / 100
    z, _, _ = geodesic(p, w1, w2, b1, b2, vae_model.encoder)
    vanilla_geodesic_list.append(z)
    
vanilla_geodesic_tensor = torch.stack(vanilla_geodesic_list, axis=0)
vanilla_geodesic_np = vanilla_geodesic_tensor.detach().cpu().numpy()

gs_geodesic_list = list()
gsvae.eval()
z0 = torch.tensor(random_gsz_test[17]).unsqueeze(0)
w1, w2, b1, b2 = gsvae.gamma(torch.tensor(z0).cuda())
one = torch.ones((1)).cuda()

for i in range(100):
    p = one * (50 - i) / 50
    z, _, _ = geodesic(p, w1, w2, b1, b2, gsvae.vae.encoder)
    gs_geodesic_list.append(z)
    
gs_geodesic_tensor = torch.stack(gs_geodesic_list, axis=0)
gs_geodesic_np = gs_geodesic_tensor.detach().cpu().numpy()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1
axs[0].contour(latent_x, latent_y, np.log(indi), levels=[0, 3], colors='k')
axs[0].contourf(latent_x, latent_y, np.log(indi), extend="both", levels=np.arange(0, 6, 0.2), cmap='gist_gray')
axs[0].scatter(anchor[0], anchor[1], c="r")
#axs[0].scatter(zs[:, 0], zs[:, 1], c='b', alpha=0.5, s=0.7)
axs[0].scatter(random_z_train[:, 0], random_z_train[:, 1], c='g', alpha=0.5, s=0.5)
axs[0].scatter(random_z_test[:, 0], random_z_test[:, 1], c='b', alpha=0.5, s=0.5)
axs[0].plot(vanilla_geodesic_np[:, 0], vanilla_geodesic_np[:, 1], 'm')
#axs[0].plot(geo_np[:, 0], geo_np[:, 1], 'm')
#axs[0].plot([geo_np[1, 0], 2 * anchor[0] - geo_np[1, 0]], [geo_np[1, 1], 2 * anchor[1] - geo_np[1, 1]], 'k-.')
axs[0].axis([-4, 4, -4, 4])
axs[0].set_title('Vanilla VAE')

# Plot 2
plt.contour(latent_x, latent_y, np.log(gsindi), levels=[0, 3], colors='k')
plt.contourf(latent_x, latent_y, np.log(gsindi), extend="both", levels=np.arange(0, 6, 0.2), cmap="gist_gray")
axs[1].scatter(anchor[0], anchor[1], c="r")
axs[1].scatter(random_gsz_train[:,0], random_gsz_train[:,1], c='g', alpha=0.5, s=0.5)
axs[1].scatter(random_gsz_test[:,0], random_gsz_test[:,1], c='b', alpha=0.5, s=0.5)
axs[1].plot(gs_geodesic_np[:, 0], gs_geodesic_np[:, 1], 'm')
#axs[1].plot(geo_np0[:, 0], geo_np0[:, 1], 'm-')
#axs[1].plot([geo_np0[0, 0], 2 * anchor[0] - geo_np0[0, 0]], [geo_np0[0, 1], 2 * anchor[1] - geo_np0[0, 1]], 'k-.')
#axs[1].plot(geo_np1[:, 0], geo_np1[:, 1], 'y')
#axs[1].plot([geo_np1[0, 0], 2 * anchor[0] - geo_np1[0, 0]], [geo_np1[0, 1], 2 * anchor[1] - geo_np1[0, 1]], 'c')
axs[1].axis([-4, 4, -4, 4])
axs[1].set_title('VAE + geodesic symmetry')

plt.show()


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

# Plot 1
axs[0].contour(latent_x, latent_y, np.log(indi), levels=np.arange(0, 5, 0.5), colors='k')
axs[0].contourf(latent_x, latent_y, np.log(indi), extend="both", levels=np.arange(0, 6, 0.2), cmap='gist_gray')
axs[0].scatter(anchor[0], anchor[1], c="r")
#axs[0].scatter(zs[:, 0], zs[:, 1], c='b', alpha=0.5, s=0.7)
axs[0].scatter(random_z_train[:, 0], random_z_train[:, 1], c='g', alpha=0.5, s=0.5)
axs[0].scatter(random_z_test[:, 0], random_z_test[:, 1], c='b', alpha=0.5, s=0.5)
axs[0].plot(vanilla_geodesic_np[:, 0], vanilla_geodesic_np[:, 1], 'm')
#axs[0].plot(geo_np[:, 0], geo_np[:, 1], 'm')
#axs[0].plot([geo_np[1, 0], 2 * anchor[0] - geo_np[1, 0]], [geo_np[1, 1], 2 * anchor[1] - geo_np[1, 1]], 'k-.')
axs[0].axis([-4, 4, -4, 4])
axs[0].set_title('Vanilla VAE')

# Plot 2
axs[1].contour(latent_x, latent_y, np.log(gsindi), levels=np.arange(0, 5, 0.5), colors='k')
axs[1].contourf(latent_x, latent_y, np.log(gsindi), extend="both", levels=np.arange(0, 6, 0.2), cmap="gist_gray")
axs[1].scatter(anchor[0], anchor[1], c="r")
axs[1].scatter(random_gsz_train[:,0], random_gsz_train[:,1], c='g', alpha=0.5, s=0.5)
axs[1].scatter(random_gsz_test[:,0], random_gsz_test[:,1], c='b', alpha=0.5, s=0.5)
axs[1].plot(gs_geodesic_np[:, 0], gs_geodesic_np[:, 1], 'm')
#axs[1].plot(geo_np0[:, 0], geo_np0[:, 1], 'm-')
#axs[1].plot([geo_np0[0, 0], 2 * anchor[0] - geo_np0[0, 0]], [geo_np0[0, 1], 2 * anchor[1] - geo_np0[0, 1]], 'k-.')
#axs[1].plot(geo_np1[:, 0], geo_np1[:, 1], 'y')
#axs[1].plot([geo_np1[0, 0], 2 * anchor[0] - geo_np1[0, 0]], [geo_np1[0, 1], 2 * anchor[1] - geo_np1[0, 1]], 'c')
axs[1].axis([-4, 4, -4, 4])
axs[1].set_title('VAE + geodesic symmetry')

plt.show()

## 12. Quantitative Symmetry Analysis

Select three test points (A, B, C) at varying distances from the anchor, compute their **geodesic reflections** through the anchor ($z' = 2 \cdot \text{anchor} - z$), and find the nearest training samples to each reflected point.

Then use `morphomnist.measure.measure_image` to extract morphological attributes (thickness, slant, etc.) and verify that the symmetry mapping preserves the correct factor relationships.

In [ ]:
# Assuming random_gsz_test is a numpy array
distances = np.linalg.norm(random_gsz_test - random_gsz_test[17], axis=1)

# Create a dictionary with key-value pairs sorted by distance values
dist_dict = {i: distance for i, distance in enumerate(distances)}
sorted_dist_dict = sorted(dist_dict.items(), key=lambda x: x[1])

sorted_dist_dict

In [ ]:
anchor = gsvae.gamma.anchor.detach().clone().cpu().numpy()

point1 = random_gsz_test[sorted_dist_dict[100][0]]
point2 = random_gsz_test[sorted_dist_dict[200][0]]
point3 = random_gsz_test[17]

point1_ref = 2 * anchor - point1
point2_ref = 2 * anchor - point2
point3_ref = 2 * anchor - point3

geo_list1 = list()
geo_list2 = list()
geo_list3 = list()

w11, w21, b11, b21 = gsvae.gamma(torch.tensor(point1).cuda().unsqueeze(0))
w12, w22, b12, b22 = gsvae.gamma(torch.tensor(point2).cuda().unsqueeze(0))
w13, w23, b13, b23 = gsvae.gamma(torch.tensor(point3).cuda().unsqueeze(0))


gsvae.eval()
one = torch.ones((1)).unsqueeze(0).cuda()

for i in range(100):
    p = one * (50 - i) / 50
    z1, _, _ = geodesic(p, w11, w21, b11, b21, gsvae.vae.encoder)
    z2, _, _ = geodesic(p, w12, w22, b12, b22, gsvae.vae.encoder)
    z3, _, _ = geodesic(p, w13, w23, b13, b23, gsvae.vae.encoder)
    
    geo_list1.append(z1)
    geo_list2.append(z2)
    geo_list3.append(z3)
    
geo_tensor1 = torch.stack(geo_list1, axis=0)
geo_np1 = geo_tensor1.detach().cpu().numpy()

geo_tensor2 = torch.stack(geo_list2, axis=0)
geo_np2 = geo_tensor2.detach().cpu().numpy()

geo_tensor3 = torch.stack(geo_list3, axis=0)
geo_np3 = geo_tensor3.detach().cpu().numpy()

point1_geo_symm = geo_np1[-1]
point2_geo_symm = geo_np2[-1]
point3_geo_symm = geo_np3[-1]

In [ ]:
image1 = test_set[sorted_dist_dict[100][0]][0].numpy().reshape(28, 28)
feature1 = test_set[sorted_dist_dict[100][0]][1]
image2 = test_set[sorted_dist_dict[200][0]][0].numpy().reshape(28, 28)
feature2 = test_set[sorted_dist_dict[200][0]][1]
image3 = test_set[17][0].numpy().reshape(28, 28)
feature3 = test_set[17][1]

In [ ]:
dist_from_point1_ref = np.linalg.norm(gsz_train - point1_ref, axis=1)
nearest_point1_ref = np.argmin(dist_from_point1_ref)
image1_ref = train_set[nearest_point1_ref][0].numpy().reshape(28, 28)
feature_ref1 = train_set[nearest_point1_ref][1]
exact_point1_ref = gsz_train[nearest_point1_ref]

dist_from_point2_ref = np.linalg.norm(gsz_train - point2_ref, axis=1)
nearest_point2_ref = np.argmin(dist_from_point2_ref)
image2_ref = train_set[nearest_point2_ref][0].numpy().reshape(28, 28)
feature_ref2 = train_set[nearest_point2_ref][1]
exact_point2_ref = gsz_train[nearest_point2_ref]

dist_from_point3_ref = np.linalg.norm(gsz_train - point3_ref, axis=1)
nearest_point3_ref = np.argmin(dist_from_point3_ref)
image3_ref = train_set[nearest_point3_ref][0].numpy().reshape(28, 28)
feature_ref3 = train_set[nearest_point3_ref][1]
exact_point3_ref = gsz_train[nearest_point3_ref]


In [ ]:
import pandas
from morphomnist.measure import measure_image

In [ ]:
area1, length1, thickness1, slant1, width1, height1 = measure_image(image1)
area2, length2, thickness2, slant2, width2, height2 = measure_image(image2)
area3, length3, thickness3, slant3, width3, height3 = measure_image(image3)

area_ref1, length_ref1, thickness_ref1, slant_ref1, width_ref1, height_ref1 = measure_image(image1_ref)
area_ref2, length_ref2, thickness_ref2, slant_ref2, width_ref2, height_ref2 = measure_image(image2_ref)
area_ref3, length_ref3, thickness_ref3, slant_ref3, width_ref3, height_ref3 = measure_image(image3_ref)

In [ ]:
from matplotlib.figure import Figure
from matplotlib.offsetbox import OffsetImage, AnnotationBbox, VPacker, TextArea

In [ ]:
vanilla_anchor = geodesic_maker.anchor.clone().detach().cpu().numpy()
gs_anchor = gsvae.gamma.anchor.clone().detach().cpu().numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(20,20))
#ax.contour(latent_x, latent_y, np.log(gsindi), extend="both", levels=np.arange(0, 6, 1), colors='k')
ax.contourf(latent_x, latent_y, np.log(gsindi), extend="both", levels=np.arange(0, 6, 0.2), cmap='gist_gray')
ax.scatter(random_gsz_train[:, 0], random_gsz_train[:, 1], c='g', alpha=0.3, s=10)
ax.scatter(random_gsz_test[:, 0], random_gsz_test[:, 1], c='b', alpha=0.3, s=10)
ax.plot(geo_np1[:, 0], geo_np1[:, 1], 'm-.', linewidth=2)
ax.plot(geo_np2[:, 0], geo_np2[:, 1], 'm-', linewidth=2)
ax.plot(geo_np3[:, 0], geo_np3[:, 1], 'm', linewidth=2)
ax.scatter(gs_anchor[0], gs_anchor[1], c='r', s=100)
ax.axis([-0.3, 0.3, -0.1, 0.5])

ax.plot([point1[0], point1_ref[0]], [point1[1], point1_ref[1]], 'k-.', linewidth=2)
ax.plot([point2[0], point2_ref[0]], [point2[1], point2_ref[1]], 'k-', linewidth=2)
ax.plot([point3[0], point3_ref[0]], [point3[1], point3_ref[1]], 'k', linewidth=2)

ax.scatter(point1[0], point1[1], c='b', s=100, marker='X')
ax.scatter(point2[0], point2[1], c='b', s=100, marker='X')
ax.scatter(point3[0], point3[1], c='b', s=100, marker='X')

ax.scatter(point1_ref[0], point1_ref[1], c='g', s=100, marker='s')
ax.scatter(point2_ref[0], point2_ref[1], c='g', s=100, marker='s')
ax.scatter(point3_ref[0], point3_ref[1], c='g', s=100, marker='s')

ax.axis('off')

fig.savefig('gs_vae_geodesic.png', dpi=100, bbox_inches='tight', pad_inches=0)

## 13. Final Annotated Figure

Overlay the test points (blue ✕), their reflections (green ■), and annotated sample images on the GSVAE latent-space metric heatmap.  
Each image is annotated with measured morphological features (thickness and slant).  
The figure is saved to `gs_vae_symm.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(20,20))
ax.contour(latent_x, latent_y, np.log(gsindi), extend="both", levels=[0,2.5], colors='k')
ax.contourf(latent_x, latent_y, np.log(gsindi), extend="both", levels=np.arange(0, 6, 0.2), cmap='gist_gray')
ax.scatter(random_gsz_train[:, 0], random_gsz_train[:, 1], c='g', alpha=0.3, s=10)
ax.scatter(random_gsz_test[:, 0], random_gsz_test[:, 1], c='b', alpha=0.3, s=10)
#ax.plot(geo_np1[:, 0], geo_np1[:, 1], 'm-.', linewidth=2)
#ax.plot(geo_np2[:, 0], geo_np2[:, 1], 'm-', linewidth=2)
#ax.plot(geo_np3[:, 0], geo_np3[:, 1], 'm', linewidth=2)
ax.scatter(gs_anchor[0], gs_anchor[1], c='r', s=100)
ax.axis([-0.4, 0.6, -0.3, 0.6])

ax.plot([point1[0], point1_ref[0]], [point1[1], point1_ref[1]], 'k-.', linewidth=2)
ax.plot([point2[0], point2_ref[0]], [point2[1], point2_ref[1]], 'k-', linewidth=2)
ax.plot([point3[0], point3_ref[0]], [point3[1], point3_ref[1]], 'k', linewidth=2)

ax.scatter(point1[0], point1[1], c='b', s=100, marker='X')
ax.scatter(point2[0], point2[1], c='b', s=100, marker='X')
ax.scatter(point3[0], point3[1], c='b', s=100, marker='X')

ax.scatter(point1_ref[0], point1_ref[1], c='g', s=100, marker='s')
ax.scatter(point2_ref[0], point2_ref[1], c='g', s=100, marker='s')
ax.scatter(point3_ref[0], point3_ref[1], c='g', s=100, marker='s')

offset1 = OffsetImage(image1, zoom=4)
offset2 = OffsetImage(image2, zoom=4)
offset3 = OffsetImage(image3, zoom=4)

offset1_ref = OffsetImage(image1_ref, zoom=4)
offset2_ref = OffsetImage(image2_ref, zoom=4)
offset3_ref = OffsetImage(image3_ref, zoom=4)

""" ab1 = AnnotationBbox(offset1, point1, xybox=[0.5, 0.15], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab2 = AnnotationBbox(offset2, point2, xybox=[0.5, -0.1], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab3 = AnnotationBbox(offset3, point3, xybox=[0.5, 0.4], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab1_ref = AnnotationBbox(offset1_ref, exact_point1_ref, xybox=[-0.3, 0.15], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab2_ref = AnnotationBbox(offset2_ref, exact_point2_ref, xybox=[-0.3, 0.4], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab3_ref = AnnotationBbox(offset3_ref, exact_point3_ref, xybox=[-0.3, -0.1], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False) """

text1 = TextArea(f"B\'\nThickness: {thickness1:.3f}\nSlant: {slant1:.3f}", textprops=dict(color='black', fontsize=20))
text2 = TextArea(f"A\'\nThickness: {thickness2:.3f}\nSlant: {slant2:.3f}", textprops=dict(color='black', fontsize=20))
text3 = TextArea(f"C\'\nThickness: {thickness3:.3f}\nSlant: {slant3:.3f}", textprops=dict(color='black', fontsize=20))

text1_ref = TextArea(f"B\nThickness: {thickness_ref1:.3f}\nSlant: {slant_ref1:.3f}", textprops=dict(color='black', fontsize=20))
text2_ref = TextArea(f"A\nThickness: {thickness_ref2:.3f}\nSlant: {slant_ref2:.3f}", textprops=dict(color='black', fontsize=20))
text3_ref = TextArea(f"C\nThickness: {thickness_ref3:.3f}\nSlant: {slant_ref3:.3f}", textprops=dict(color='black', fontsize=20))

thickness_box = TextArea("Thickness\nA ⇔ B ~ C → A\' ⇔ B\' ~ C\'", textprops=dict(color='black', fontsize=20))
slant_box = TextArea("Slant\nA ~ B ⇔ C → A\' ~ B\' ⇔ C\'", textprops=dict(color='black', fontsize=20))

vp1 = VPacker(children=[offset1, text1])
vp2 = VPacker(children=[offset2, text2])
vp3 = VPacker(children=[offset3, text3])
vp1_ref = VPacker(children=[offset1_ref, text1_ref])
vp2_ref = VPacker(children=[offset2_ref, text2_ref])
vp3_ref = VPacker(children=[offset3_ref, text3_ref])

vp_textbox = VPacker(children=[thickness_box, slant_box])

ab1 = AnnotationBbox(vp1, point1, xybox=[0.5, 0.15], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab2 = AnnotationBbox(vp2, point2, xybox=[0.5, -0.1], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab3 = AnnotationBbox(vp3, point3, xybox=[0.5, 0.4], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)

ab1_ref = AnnotationBbox(vp1_ref, exact_point1_ref, xybox=[-0.3, 0.15], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab2_ref = AnnotationBbox(vp2_ref, exact_point2_ref, xybox=[-0.3, 0.4], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)
ab3_ref = AnnotationBbox(vp3_ref, exact_point3_ref, xybox=[-0.3, -0.1], arrowprops=dict(arrowstyle="-|>", lw=0.5), frameon=False)

ab_tb = AnnotationBbox(vp_textbox, [0.1, 0.55], frameon=True, bboxprops=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.5'))

ax.add_artist(ab1)
ax.add_artist(ab2)
ax.add_artist(ab3)
ax.add_artist(ab1_ref)
ax.add_artist(ab2_ref)
ax.add_artist(ab3_ref)
ax.add_artist(ab_tb)

ax.axis('off')

fig.savefig('gs_vae_symm.png', dpi=100, bbox_inches='tight', pad_inches=0)
